# Capítulo 5 — Orquestração

**Livro:** começa na página **148** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## Agent Types · PDF p. 149

#### 🧪 Exercício: ReAct com um modelo falso

Para testar a orquestração **sem gastar**, crie `ModeloFalso` que devolve respostas roteirizadas no formato ReAct ("Pensamento: ... / Ação: ferramenta[arg]" e por fim "Resposta: ..."). Escreva o laço que interpreta a ação com regex, executa a ferramenta e devolve a "Observação".

In [ ]:
CATALOGO = {"teclado": 199.9, "mouse": 89.9, "monitor": 1299.0, "cadeira": 899.0}
PEDIDOS = {"A100": {"status": "enviado", "previsao": "2025-06-12"}, "A101": {"status": "em separação", "previsao": "2025-06-15"}}
def consultar_preco(produto: str) -> str:
    """Devolve o preço de um produto do catálogo."""
    p = CATALOGO.get(produto.lower())
    return f"{produto}: R$ {p:.2f}" if p else f"{produto} não está no catálogo"
def status_pedido(pedido_id: str) -> str:
    """Devolve o status e a previsão de entrega de um pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    return json.dumps(p, ensure_ascii=False) if p else "pedido não encontrado"
FUNCOES = {"consultar_preco": consultar_preco, "status_pedido": status_pedido}
TOOLS = [{"type": "function", "function": {"name": "consultar_preco", "description": "Preço de um produto do catálogo",
          "parameters": {"type": "object", "properties": {"produto": {"type": "string"}}, "required": ["produto"]}}},
         {"type": "function", "function": {"name": "status_pedido", "description": "Status e previsão de entrega de um pedido",
          "parameters": {"type": "object", "properties": {"pedido_id": {"type": "string"}}, "required": ["pedido_id"]}}}]
class ModeloFalso:
    def __init__(self, roteiro): self.roteiro = iter(roteiro)
    def __call__(self, historico): return next(self.roteiro)
modelo = ModeloFalso(["Pensamento: preciso do preço.\nAção: consultar_preco[monitor]",
                      "Pensamento: agora o pedido.\nAção: status_pedido[A101]",
                      "Resposta: O monitor custa R$ 1.299,00 e o pedido A101 está em separação."])
historico = "Pergunta: preço do monitor e status do A101?"
for passo in range(5):
    saida = modelo(historico); print(saida)
    if saida.startswith("Resposta:"): break
    nome, arg = re.search(r"Ação: (\w+)\[(.+?)\]", saida).groups()
    obs = FUNCOES[nome](arg); print("Observação:", obs); historico += f"\n{saida}\nObservação: {obs}"

## Tool Selection · PDF p. 156

#### 🧪 Exercício: Escolher ferramentas por similaridade

Com 15 ferramentas descritas em texto, mandar todas ao modelo gasta contexto. Selecione as 3 mais relevantes para o pedido do usuário por similaridade TF-IDF entre o pedido e as descrições.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
desc = {"consultar_preco": "preço valor custo de produto do catálogo", "status_pedido": "status entrega rastreio de pedido",
        "calcular_frete": "frete entrega CEP peso", "abrir_chamado": "abrir chamado suporte problema defeito",
        "trocar_endereco": "alterar endereço de entrega", "listar_cupons": "cupom desconto promoção",
        "cancelar_pedido": "cancelar pedido compra", "nota_fiscal": "segunda via nota fiscal", "estoque": "disponibilidade estoque produto",
        "garantia": "garantia defeito prazo", "pontos": "pontos fidelidade saldo", "horario_loja": "horário funcionamento loja física",
        "parcelamento": "parcelas cartão juros", "avaliacoes": "avaliações opiniões de clientes", "comparar": "comparar dois produtos"}
vec = TfidfVectorizer().fit(desc.values()); M = vec.transform(desc.values())
pedido = "meu teclado chegou com defeito, quero acionar a garantia"
sim = (M @ vec.transform([pedido]).T).toarray().ravel()
print([list(desc)[i] for i in sim.argsort()[::-1][:3]])

## Tool Execution · PDF p. 175

#### 🧪 Exercício: Repetir com espera exponencial

Escreva o decorador `@com_retentativas(max_tentativas=4)` que repete uma ferramenta que falhou, esperando 0,1·2ⁿ s entre tentativas, e desiste com uma mensagem clara. Teste com uma ferramenta que falha 2 vezes e depois funciona.

In [ ]:
import functools
def com_retentativas(max_tentativas=4, base=0.1):
    def deco(f):
        @functools.wraps(f)
        def envolta(*a, **k):
            for n in range(max_tentativas):
                try: return f(*a, **k)
                except Exception as e:
                    print(f"tentativa {n + 1} falhou: {e}"); time.sleep(base * 2 ** n)
            return f"ERRO: {f.__name__} falhou {max_tentativas} vezes"
        return envolta
    return deco
falhas = {"n": 0}
@com_retentativas()
def api_instavel():
    falhas["n"] += 1
    if falhas["n"] <= 2: raise TimeoutError("tempo esgotado")
    return "ok na tentativa " + str(falhas["n"])
print(api_instavel())

## Tool Topologies · PDF p. 176

#### 🧪 Exercício: Ferramentas em paralelo e em cadeia

Monte uma topologia: duas ferramentas independentes em **paralelo** (preço do teclado e do mouse) e uma terceira em **sequência** que depende das duas (soma com 10% de desconto no combo).

In [ ]:
CATALOGO = {"teclado": 199.9, "mouse": 89.9, "monitor": 1299.0, "cadeira": 899.0}
PEDIDOS = {"A100": {"status": "enviado", "previsao": "2025-06-12"}, "A101": {"status": "em separação", "previsao": "2025-06-15"}}
def consultar_preco(produto: str) -> str:
    """Devolve o preço de um produto do catálogo."""
    p = CATALOGO.get(produto.lower())
    return f"{produto}: R$ {p:.2f}" if p else f"{produto} não está no catálogo"
def status_pedido(pedido_id: str) -> str:
    """Devolve o status e a previsão de entrega de um pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    return json.dumps(p, ensure_ascii=False) if p else "pedido não encontrado"
FUNCOES = {"consultar_preco": consultar_preco, "status_pedido": status_pedido}
TOOLS = [{"type": "function", "function": {"name": "consultar_preco", "description": "Preço de um produto do catálogo",
          "parameters": {"type": "object", "properties": {"produto": {"type": "string"}}, "required": ["produto"]}}},
         {"type": "function", "function": {"name": "status_pedido", "description": "Status e previsão de entrega de um pedido",
          "parameters": {"type": "object", "properties": {"pedido_id": {"type": "string"}}, "required": ["pedido_id"]}}}]
from concurrent.futures import ThreadPoolExecutor
def preco_num(p): return CATALOGO[p]
with ThreadPoolExecutor() as ex:
    teclado, mouse = ex.map(preco_num, ["teclado", "mouse"])
def combo(a, b, desconto=0.10): return round((a + b) * (1 - desconto), 2)
print(teclado, mouse, "-> combo:", combo(teclado, mouse))

## Context Engineering · PDF p. 186

#### 🧪 Exercício: Montar o contexto dentro do orçamento

Escreva `montar_contexto(partes, orcamento)` que recebe partes com prioridade (instruções > pergunta > documentos > histórico) e inclui o máximo possível sem passar do orçamento de tokens, cortando primeiro o que tem menos prioridade.

In [ ]:
partes = [("instrucoes", 1, "Responda só com base nos documentos. " * 3), ("pergunta", 2, "Qual o prazo de troca?"),
          ("documento", 3, "Trocas em até 30 dias com nota fiscal. " * 8), ("historico", 4, "usuário: oi / assistente: olá " * 20)]
def montar_contexto(partes, orcamento):
    usado, incluidas = 0, []
    for nome, _, texto in sorted(partes, key=lambda p: p[1]):
        t = contar_tokens(texto)
        if usado + t <= orcamento: incluidas.append(nome); usado += t
    return incluidas, usado
for orc in [60, 150, 400]: print(orc, montar_contexto(partes, orc))